# Mineração de Dados — PL7 (Knowledge Graphs)

#### Nome: Fernando Jorge Silva Pires

#### Numero: PG60253


In [3]:
# Instale as dependências necessárias
# !pip install neo4j spacy pandas
# !python -m spacy download pt_core_news_sm

from neo4j import GraphDatabase
import spacy
import pandas as pd
from typing import List, Dict, Tuple
import re

# Carregue o modelo NLP em português
nlp = spacy.load("pt_core_news_sm")

## Parte 1 – Conexão ao Neo4j

### Questão 1.1. Configure a conexão ao Neo4j.

In [4]:
class Neo4jConnection:
    def __init__(self, uri, user, password):
        # Inicializa o driver do Neo4j com os parâmetros fornecidos
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self):
        # Fecha a conexão com o Neo4j
        self.driver.close()

    def query(self, query, parameters=None):
        # Executa uma query Cypher e retorna os resultados como lista
        with self.driver.session() as session:
            result = session.run(query, parameters or {})
            return [record.data() for record in result]


# Configure a conexão com os valores corretos
conn = Neo4jConnection("bolt://localhost:7687", "neo4j", "password")

# Teste a conexão executando uma query simples
try:
    result = conn.query("RETURN 'Conexão bem-sucedida!' as message")
    print(result[0]["message"])
except Exception as e:
    print(f"Erro na conexão: {e}")

Erro na conexão: Couldn't connect to localhost:7687 (resolved to ('[::1]:7687', '127.0.0.1:7687')):
Failed to establish connection to ResolvedIPv6Address(('::1', 7687, 0, 0)) (reason [WinError 10061] No connection could be made because the target machine actively refused it)
Failed to establish connection to ResolvedIPv4Address(('127.0.0.1', 7687)) (reason [WinError 10061] No connection could be made because the target machine actively refused it)


## Parte 2 – NLP para Extração de Entidades

### Questão 2.1. Extraia entidades com spaCy

In [5]:
def extract_entities(text: str) -> List[Tuple[str, str]]:
    """
    Retorna lista de tuplas (entidade, tipo) extraídas com spaCy.
    """
    doc = nlp(text)
    entities = []

    # Percorre as entidades reconhecidas no documento e adiciona à lista
    for ent in doc.ents:
        entities.append((ent.text, ent.label_))

    return entities


# Texto de exemplo
texto_exemplo = """
A OpenAI lançou o ChatGPT em novembro de 2022. Sam Altman criou a empresa.
O ChatGPT utiliza aprendizagem profunda e foi desenvolvido em São Francisco.
"""

# Teste a função com o texto_exemplo
entities = extract_entities(texto_exemplo)
print("Entidades encontradas:")
for entity, label in entities:
    print(f"  - {entity} ({label})")

Entidades encontradas:
  - OpenAI (ORG)
  - ChatGPT (MISC)
  - Sam Altman (PER)
  - ChatGPT (MISC)
  - São Francisco (LOC)


### Questão 2.2. Extração de relações simples

In [6]:
def extract_relations(text: str) -> List[Tuple[str, str, str]]:
    """
    Extrai relações simples baseadas em padrões sintáticos.
    Retorna lista de triplas (sujeito, relação, objeto).
    """
    doc = nlp(text)
    relations = []

    for token in doc:
        # Verifica se o token é sujeito nominal
        if token.dep_ == "nsubj":
            subject = token.text
            verb = token.head  # Verbo associado ao sujeito

            # Procura objeto direto entre os filhos do verbo
            for child in verb.children:
                if child.dep_ in ("obj", "dobj"):
                    relations.append((subject, verb.lemma_, child.text))

    return relations


# Teste a função com o texto_exemplo
relations = extract_relations(texto_exemplo)
print("Relações encontradas:")
for subj, rel, obj in relations:
    print(f"  - {subj} --[{rel}]--> {obj}")

Relações encontradas:
  - OpenAI --[lançar]--> ChatGPT
  - Sam --[criar]--> empresa
  - ChatGPT --[utilizar]--> aprendizagem


### Questão 2.3. Limitações e melhorias da abordagem baseada em dependências sintáticas

**Limitações:**

1. **Cobertura reduzida**: apenas captura relações do tipo sujeito–verbo–objeto direto. Relações expressas através de preposições (e.g., *"fundou em 2002"*), passivas ou estruturas nominais não são extraídas.
2. **Dependência da qualidade do parser**: erros de parsing propagam-se diretamente para as triplas extraídas — frases longas ou com estrutura complexa tendem a ser mal analisadas.
3. **Sem desambiguação de entidades**: o mesmo referente pode aparecer com variações de texto ("OpenAI", "a empresa") e ser tratado como entidades distintas.
4. **Relações implícitas**: o modelo não consegue inferir relações que não estão explicitamente codificadas na sintaxe da frase.
5. **Dependência de idioma/modelo**: o modelo `pt_core_news_sm` é pequeno e treinado em corpus limitado, com menor desempenho em textos de domínio específico.

**Melhorias possíveis:**

- Utilizar um modelo maior (`pt_core_news_lg`) ou um modelo de linguagem (e.g., BERT/RoBERTa) para extração de relações por classificação.
- Adicionar resolução de correferências para ligar pronomes e expressões nominais ao mesmo nó.
- Expandir os padrões sintáticos para cobrir frases preposicionadas, relativas e construções passivas.
- Aplicar *Named Entity Linking* (NEL) para normalizar entidades a identificadores canónicos (e.g., Wikidata QIDs).

## Parte 3 – Construção do Knowledge Graph

### Questão 3.1. Criação de Nós e Relacionamentos

In [7]:
def create_knowledge_graph(conn: Neo4jConnection, entities: List[Tuple[str, str]],
                           relations: List[Tuple[str, str, str]]):
    """
    Cria nós e relacionamentos no Neo4j baseado nas entidades e relações extraídas.
    """
    # Cria constraint de unicidade no nome da entidade (evita duplicação)
    conn.query(
        "CREATE CONSTRAINT entity_name_unique IF NOT EXISTS "
        "FOR (e:Entity) REQUIRE e.name IS UNIQUE"
    )

    # Para cada entidade, cria um nó com MERGE (evita duplicados)
    for name, ent_type in entities:
        conn.query(
            "MERGE (e:Entity {name: $name}) SET e.type = $type",
            {"name": name, "type": ent_type}
        )

    # Para cada relação, encontra os nós sujeito e objeto e cria o relacionamento
    for subj, rel, obj in relations:
        # Normaliza o nome da relação para ser um tipo Cypher válido
        rel_type = re.sub(r"[^A-Za-z0-9_]", "_", rel.upper())
        conn.query(
            f"MATCH (a:Entity {{name: $subj}}), (b:Entity {{name: $obj}}) "
            f"MERGE (a)-[:{rel_type}]->(b)",
            {"subj": subj, "obj": obj}
        )


# Aplica a função com as entidades e relações extraídas anteriormente
try:
    create_knowledge_graph(conn, entities, relations)
    print("Knowledge graph criado com sucesso!")
except Exception as e:
    print(f"Erro ao criar knowledge graph: {e}")

Erro ao criar knowledge graph: Couldn't connect to localhost:7687 (resolved to ('[::1]:7687', '127.0.0.1:7687')):
Failed to establish connection to ResolvedIPv6Address(('::1', 7687, 0, 0)) (reason [WinError 10061] No connection could be made because the target machine actively refused it)
Failed to establish connection to ResolvedIPv4Address(('127.0.0.1', 7687)) (reason [WinError 10061] No connection could be made because the target machine actively refused it)


### Questão 3.2. Consultas Cypher através do Python

In [8]:
def query_graph(conn: Neo4jConnection, entity_name: str):
    """
    Consulta o knowledge graph para encontrar conexões de uma entidade.
    Retorna: nome da entidade, lista de entidades conectadas e tipos de relação.
    """
    query = """
    MATCH (e:Entity {name: $name})
    OPTIONAL MATCH (e)-[r]-(other:Entity)
    RETURN e.name AS entity,
           collect(DISTINCT other.name) AS connections,
           collect(DISTINCT type(r))   AS relation_types
    """

    results = conn.query(query, {"name": entity_name})
    return results[0] if results else None


# Teste a consulta com uma entidade do grafo
try:
    info = query_graph(conn, "OpenAI")
    if info:
        print(f"Entidade: {info['entity']}")
        print(f"Conexões: {info['connections']}")
        print(f"Tipos de relação: {info['relation_types']}")
    else:
        print("Entidade não encontrada no grafo.")
except Exception as e:
    print(f"Erro na consulta: {e}")

Erro na consulta: Couldn't connect to localhost:7687 (resolved to ('[::1]:7687', '127.0.0.1:7687')):
Failed to establish connection to ResolvedIPv6Address(('::1', 7687, 0, 0)) (reason [WinError 10061] No connection could be made because the target machine actively refused it)
Failed to establish connection to ResolvedIPv4Address(('127.0.0.1', 7687)) (reason [WinError 10061] No connection could be made because the target machine actively refused it)


### Questão 3.3. MERGE vs CREATE e importância das constraints

**MERGE em vez de CREATE:**

`MERGE` funciona como um "upsert": verifica se o nó (ou relação) já existe e, caso exista, não o duplica — caso contrário, cria-o. Se usássemos `CREATE`, cada execução do pipeline geraria nós duplicados para a mesma entidade, corrompendo o grafo com múltiplos nós para "OpenAI", "Sam Altman", etc. Em pipelines que processam documentos incrementalmente, o uso de `MERGE` é essencial para manter a consistência do grafo.

**Importância das constraints:**

As constraints de unicidade (`UNIQUE`) no Neo4j garantem, a nível da base de dados, que dois nós com o mesmo valor numa propriedade não coexistam. Isso tem dois benefícios:

1. **Integridade dos dados**: mesmo que o código falhe ou seja executado de forma concorrente, a constraint impede a introdução de duplicados.
2. **Desempenho**: o Neo4j cria automaticamente um índice para a propriedade restrita, tornando as operações `MERGE` e `MATCH` por essa propriedade muito mais rápidas — especialmente relevante em grafos com milhões de nós.

## Parte 4 – Processamento de Múltiplos Documentos

### Questão 4.1. Processamento em lote

In [9]:
# Documentos de exemplo
documentos = [
    {
        "id": 1,
        "texto": "Elon Musk fundou a SpaceX em 2002. A empresa lançou o Falcon 9.",
        "fonte": "Wikipedia"
    },
    {
        "id": 2,
        "texto": "Tesla, liderada por Elon Musk, produz veículos elétricos.",
        "fonte": "Notícia"
    },
    {
        "id": 3,
        "texto": "SpaceX e Tesla colaboram em projetos de tecnologia sustentável.",
        "fonte": "Artigo"
    }
]


def process_documents(conn: Neo4jConnection, documents: List[Dict]):
    """
    Processa múltiplos documentos e enriquece o knowledge graph.
    Para cada documento:
      - Extrai entidades e relações do texto;
      - Adiciona entidades e relações ao grafo;
      - Cria um nó Document e relacionamentos MENTIONED_IN.
    """
    for doc in documents:
        texto = doc["texto"]

        # Extrai entidades e relações do texto do documento
        doc_entities = extract_entities(texto)
        doc_relations = extract_relations(texto)

        # Adiciona entidades e relações ao knowledge graph
        create_knowledge_graph(conn, doc_entities, doc_relations)

        # Cria nó Document com id, source e text
        conn.query(
            "MERGE (d:Document {id: $id}) SET d.source = $source, d.text = $text",
            {"id": doc["id"], "source": doc["fonte"], "text": texto}
        )

        # Para cada entidade, cria relacionamento MENTIONED_IN com o documento
        for name, _ in doc_entities:
            conn.query(
                "MATCH (e:Entity {name: $name}), (d:Document {id: $doc_id}) "
                "MERGE (e)-[:MENTIONED_IN]->(d)",
                {"name": name, "doc_id": doc["id"]}
            )


# Processa os documentos
try:
    process_documents(conn, documentos)
    print("Documentos processados e integrados ao grafo!")
except Exception as e:
    print(f"Erro ao processar documentos: {e}")

Erro ao processar documentos: Couldn't connect to localhost:7687 (resolved to ('[::1]:7687', '127.0.0.1:7687')):
Failed to establish connection to ResolvedIPv6Address(('::1', 7687, 0, 0)) (reason [WinError 10061] No connection could be made because the target machine actively refused it)
Failed to establish connection to ResolvedIPv4Address(('127.0.0.1', 7687)) (reason [WinError 10061] No connection could be made because the target machine actively refused it)


### Questão 4.2. Análise de Centralidade

In [10]:
def find_important_entities(conn: Neo4jConnection):
    """
    Encontra as entidades mais conectadas no grafo (centralidade de grau).
    Retorna as 5 entidades com mais conexões distintas.
    """
    query = """
    MATCH (e:Entity)-[r]-(other:Entity)
    RETURN e.name AS entity,
           e.type  AS type,
           count(DISTINCT other) AS connections
    ORDER BY connections DESC
    LIMIT 5
    """

    results = conn.query(query)

    print("Entidades mais importantes (centralidade de grau):")
    for row in results:
        print(f"  [{row['type']}] {row['entity']} — {row['connections']} conexão(ões)")

    return results


# Executa a análise de importância
try:
    find_important_entities(conn)
except Exception as e:
    print(f"Erro na análise de centralidade: {e}")

Erro na análise de centralidade: Couldn't connect to localhost:7687 (resolved to ('[::1]:7687', '127.0.0.1:7687')):
Failed to establish connection to ResolvedIPv6Address(('::1', 7687, 0, 0)) (reason [WinError 10061] No connection could be made because the target machine actively refused it)
Failed to establish connection to ResolvedIPv4Address(('127.0.0.1', 7687)) (reason [WinError 10061] No connection could be made because the target machine actively refused it)
